# 🚀 AI Designer Backend — Colab Edition v2

**Khắc phục:** Phiên bản này dùng **subprocess.Popen** thay vì threading để server chạy ổn định hơn.

**Nếu Colab bị disconnect:** Chỉ cần reconnect và chạy lại từ cell 1.

---

### CELL 1: Mount Google Drive (LƯU TRỮ BỀN VỮNG)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

---

### CELL 2: Clone project lên Drive (CHẠY 1 LẦN THÔI)

In [ ]:
# Chạy cell này CHỈ 1 LẦN để clone repo
# Sau đó comment lại để tránh ghi đè

# THAY_THẾ_URL bằng link Git repo của bạn
GIT_URL = 'https://github.com/YOUR_USERNAME/AI_GEN.git'  # @param {type:"string"}

!git clone $GIT_URL /content/drive/MyDrive/AI_GEN 2>&1 | tail -5
print('✅ Project cloned to Google Drive')

---

### CELL 3: Cài đặt Dependencies (CHẠY 1 LẦN, sau đó comment)

In [ ]:
# CÀI ĐẶT LẠI SAU KHI RECONNECT COLAB
# (Colab xóa pip packages khi runtime disconnect)

!pip install torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121 -q

!pip install fastapi uvicorn[standard] python-multipart aiofiles pydantic python-dotenv -q
!pip install diffusers transformers accelerate peft safetensors huggingface_hub -q
!pip install opencv-python opencv-contrib-python scikit-image scipy Pillow numpy tqdm -q
!pip install open-clip-torch -q
!pip install pyngrok -q

print('✅ Dependencies installed')

---

### CELL 4: Setup Token & ngrok (MỖI LẦN RECONNECT

In [ ]:
import os
import getpass

# HuggingFace Token
HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('🔑 HuggingFace Token: ')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGINGFACE_TOKEN'] = HF_TOKEN

# ngrok Authtoken
NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass('🔑 ngrok Authtoken: ')

from pyngrok import conf
conf.get_default().auth_token = NGROK_TOKEN

print('✅ Tokens configured')

---

### CELL 5: Tạo .env config (MỖI LẦN RECONNECT)

In [ ]:
# Tạo .env cho backend (ghi ra Google Drive để lưu bền vững)
PROJECT_DIR = '/content/drive/MyDrive/AI_GEN'

env_content = f"""
INFERENCE_BACKEND=cuda
GPU_MODEL=stabilityai/sdxl-turbo
CPU_MODEL=segmind/tiny-sd
LORA_AUTO_LOAD=true
LOCAL_INFER_PROFILE=fast
COMPILE_ON_LOAD=false
HF_TOKEN={os.environ.get('HF_TOKEN', '')}
"""

os.makedirs(PROJECT_DIR, exist_ok=True)
with open(f'{PROJECT_DIR}/.env', 'w') as f:
    f.write(env_content.strip())

print(f'✅ .env created at {PROJECT_DIR}/.env')

---

### CELL 6: Fix CORS trong main.py (CHẠY 1 LẦN)

In [ ]:
# Sửa CORS để cho phép mọi origin (cần thiết cho cloud frontend)
PROJECT_DIR = '/content/drive/MyDrive/AI_GEN'

main_py = f'{PROJECT_DIR}/app/main.py'

if os.path.exists(main_py):
    with open(main_py, 'r') as f:
        content = f.read()

    # Thay thế CORS config
    old = '''allow_origins=["http://localhost:5173", "http://localhost:3000"],\n    allow_credentials=True,'''
    new = '''allow_origins=["*"],\n    allow_credentials=False,'''

    if old in content:
        content = content.replace(old, new)
        with open(main_py, 'w') as f:
            f.write(content)
        print('✅ CORS fixed in main.py')
    else:
        print('⚠️  CORS already fixed or pattern not found')
else:
    print(f'❌ main.py not found at {main_py}')
    print('   Hãy clone repo vào Google Drive trước (cell 2)')

---

### CELL 7: Khởi động Backend + ngrok (CELL NÀY GIỮ CHẠY MÃI)

In [ ]:
# ⭐ CELL QUAN TRỌNG NHẤT — GIỮ CELL NÀY CHẠY!
# Đây là cell cuối cùng. Sau khi chạy xong:
# 1. Copy URL ngrok bên dưới
# 2. ĐỪNG STOP cell này!

import os
import subprocess
import time
import urllib.request
from pyngrok import ngrok

PROJECT_DIR = '/content/drive/MyDrive/AI_GEN'
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = PROJECT_DIR

# Kill port 8000 nếu có process cũ
!pkill -f 'uvicorn' 2>/dev/null
time.sleep(2)

# Kill tất cả ngrok tunnels cũ
try:
    ngrok.kill()
except:
    pass
time.sleep(1)

# Mở ngrok tunnel TRƯỚC khi start server
print('🔓 Opening ngrok tunnel...')
http_tunnel = ngrok.connect(8000, bind_tls=True)
public_url = http_tunnel.public_url
api_url = public_url + '/api'

print()
print('=' * 65)
print('🎉 BACKEND PUBLIC URL:')
print(f'   {public_url}')
print(f'   API Base: {api_url}')
print('=' * 65)
print()
print('⏳ Starting FastAPI server...')
print('   (Lần đầu sẽ tải SDXL Turbo ~6GB — đợi 3-10 phút)')
print()
# Start uvicorn bằng subprocess — CHẠY NGẦM không blocking
log_file = open(f'{PROJECT_DIR}/backend.log', 'w')

proc = subprocess.Popen(
    [
        'python', '-m', 'uvicorn',
        'app.main:app',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--log-level', 'info'
    ],
    cwd=PROJECT_DIR,
    env={**os.environ, 'PYTHONPATH': PROJECT_DIR},
    stdout=log_file,
    stderr=subprocess.STDOUT
)

print(f'   Server PID: {proc.pid}')
print(f'   Log file: {PROJECT_DIR}/backend.log')
print()
print('⏳ Đang đợi server khởi động...')
print('   (Xem log: !tail -f /content/drive/MyDrive/AI_GEN/backend.log)')
print()

---

### CELL 8: Kiểm tra server & log (CHẠY SONG SONG VỚI CELL 7)

In [ ]:
# Chạy cell này để kiểm tra server có hoạt động không
# Có thể chạy nhiều lần (trong khi cell 7 vẫn chạy)

import time
import urllib.request
import json

public_url = None

# Đọc URL từ log file nếu có
try:
    with open(f'{PROJECT_DIR}/backend.log', 'r') as f:
        log = f.read()
    # Tìm ngrok URL trong log
    import re
    match = re.search(r'https://[^\s]+\.ngrok-free\.app', log)
    if match:
        public_url = match.group()
except:
    pass

if not public_url:
    print('⚠️  Chưa có ngrok URL — chạy cell 7 trước')
else:
    print(f'🔗 Backend: {public_url}')
    print()

    # Test health
    print('📡 Testing health endpoint...')
    for i in range(5):
        try:
            resp = urllib.request.urlopen(f'{public_url}/health', timeout=10)
            data = json.loads(resp.read())
            print('✅ Server ĐANG CHẠY!')
            print(f'   Backend: {data.get("backend")}')
            print(f'   GPU available: {data.get("gpu_available")}')
            break
        except Exception as e:
            print(f'   Thử {i+1}/5: {str(e)[:60]}...')
            time.sleep(10)
    else:
        print()
        print('⚠️  Server chưa reply sau 50s')
        print('   → Xem log bằng lệnh bên dưới:')
        print(f'   !tail -50 {PROJECT_DIR}/backend.log')

    print()
    # Tail log
    print(f'📋 Backend log (cuối 30 dòng):')
    try:
        with open(f'{PROJECT_DIR}/backend.log', 'r') as f:
            lines = f.readlines()
        for line in lines[-30:]:
            print('  ' + line.rstrip())
    except Exception as e:
        print(f'   Không đọc được log: {e}')

---

## Cách sử dụng — QUY TRÌNH ĐẦY ĐỦ

### Lần đầu tiên (clone + setup):
1. Cell 1: Mount Drive
2. Cell 2: Clone repo (THAY URL!)
3. Cell 3: Cài dependencies
4. Cell 6: Fix CORS

### Mỗi lần reconnect Colab (runtime bị mất):
1. Cell 1: Mount Drive
2. Cell 3: Cài dependencies (Colab xóa pip khi disconnect)
3. Cell 4: Nhập tokens
4. Cell 5: Tạo .env
5. **Cell 7: KHỞI ĐỘNG BACKEND** ← giữ chạy!
6. Cell 8: Kiểm tra server

### Sau khi backend chạy thành công:
- Copy URL ngrok (VD: `https://xxxx.ngrok-free.app`)
- Frontend: Settings → Backend URL → dán → Kết nối
- Generate!

---

## Xem log khi có lỗi
```bash
!tail -f /content/drive/MyDrive/AI_GEN/backend.log
```